# Azure OpenAI Audio Evaluation Demo

A clean demonstration of Azure OpenAI audio evaluation workflow.

## Prerequisites
- Azure OpenAI service with audio model deployment
- Azure CLI authenticated with access to the Azure AI project

In [1]:
from dotenv import load_dotenv
from scripts.eval_utils import AsyncEvalClient

# Load environment variables
load_dotenv()

# Setup evaluation client with Azure CLI authentication
PROJECT_ENDPOINT = "https://eastus2-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus2-prakharg-demo-2026"
client = AsyncEvalClient(project_endpoint=PROJECT_ENDPOINT)
print("🎉 Azure OpenAI Evaluation Client ready!")

AUDIO_MODEL_DEPLOYMENT_NAME = "gpt-audio"
AUDIO_GRADER_MODEL_DEPLOYMENT_NAME = "gpt-4o-mini"

🎉 Azure OpenAI Evaluation Client ready!


## Load and Create Audio Dataset

Load audio samples from HuggingFace and create an evaluation dataset file in JSONL format. The function will:
- Download the audio dataset from hugging face
- Convert audio to base64-encoded WAV format
- Prepare the file for evaluation.
- Display first 3 rows from the dataset.

In [2]:
from scripts.audio_utils import load_and_create_audio_dataset, display_items

# Load dataset from Hugging Face and create the audio dataset for evaluation
load_and_create_audio_dataset("AbstractTTS/CREMA-D", max_samples=10)

# Display the created evaluation file
display_items(3)

C:\Users\prakharg\.copilot\session-state\f1833604-4ea0-4404-893e-2421eb3ad912\files\evaluation-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Created evaluation file with 10 items
{
  "item": {
    "audio_data": "...",
    "expected_emotion": "anger"
  }
}
{
  "item": {
    "audio_data": "...",
    "expected_emotion": "disgust"
  }
}
{
  "item": {
    "audio_data": "...",
    "expected_emotion": "fear"
  }
}


In [3]:
# Upload the evaluation file to Azure OpenAI
eval_file_id = await client.upload_file(
    file_name="audio_emotion_evaluation.jsonl",
    file_path="./data/audio_emotion_evaluation.jsonl")
print(f"✅ Eval file ID: {eval_file_id}")

File 'audio_emotion_evaluation.jsonl' already exists. Returning existing file ID.
✅ Eval file ID: file-25dd9bff31d741209fb9647910e9ce8b


## Upload Evaluation Dataset

Upload the prepared audio evaluation file to your Azure OpenAI account. This file contains base64-encoded audio samples with expected emotions that will be used as the data source for running evaluations.

## Initialize Azure OpenAI Client



In [4]:
score_model = {
      "type": "score_model",
      "name": "Tone/Emotion Grader",
      "model": AUDIO_GRADER_MODEL_DEPLOYMENT_NAME,
      "input": [
        {
          "role": "system",
          "content": "You are a strict grader. Compare the model's predicted emotion with {{item.expected_emotion}}. Return only a float from 0.0 to 1.0, where 1.0 is an exact semantic match and 0.0 is incorrect."
        },
        {
          "role": "user",
          "content": "Expected emotion: {{item.expected_emotion}}\nModel response: {{sample.output_text}}"
        }
      ],
      "range": [
        0,
        1
      ],
      "pass_threshold": 0.5
    }

eval_id = await client.create_eval_sdk(
    name="Audio Emotion Evaluation",
    testing_criteria=[score_model],
    data_source_config={
    "type": "custom",
    "item_schema": {
      "type": "object",
      "properties": {
        "audio_data": {
          "type": "string",
          "description": "Base64-encoded WAV audio data."
        },
        "expected_emotion": {
          "type": "string",
          "description": "The expected primary emotion in the audio."
        }
      },
      "required": [
        "audio_data",
        "expected_emotion"
      ]
    },
    "include_sample_schema": True,
  })

Evaluation created successfully with ID: eval_a4b89507d27c4dc2a429244a03475469


## Create Evaluation with Audio Emotion Grader

Define a score model grader that evaluates the model's emotion classification from audio input. The grader:
- Compares the model's text classification with the expected emotion
- Compares it against the expected emotion from the dataset
- Returns a similarity score from 0.0 (no match) to 1.0 (exact match)
- Sets a passing threshold of 0.5 for acceptable emotion matching

## Run the Evaluation

Configure the data source and input messages for the evaluation run. This defines:
- The audio model to use for emotion identification
- System prompt instructing the model to analyze audio emotions
- User instructions to identify emotions from the audio input
- How to format multi-modal messages with text and audio
- Temperature settings for text classification output

In [5]:
import json

with open("./data/audio_emotion_evaluation.jsonl", encoding="utf-8") as f:
    eval_items = [json.loads(line) for line in f]

data_source = {
    "type": "completions",
    "model": AUDIO_MODEL_DEPLOYMENT_NAME,
    "sampling_params": {
      "temperature": 0.8
    },
    "source": {
      "type": "file_content",
      "content": eval_items
    },
    "input_messages": {
      "type": "template",
      "template": [
        {
          "role": "system",
          "content": "You are a assistant that tells the emotion of audio input. You will be given an audio input."
        },
        {
          "role": "user",
          "type": "message",
          "content": {
            "type": "input_text",
            "text": "Listen to the audio and identify the primary emotion. Respond with exactly one word from: anger, fear, disgust, happy, neutral, sad."
          }
        },
        {
          "role": "user",
          "type": "message",
          "content": {
            "type": "input_audio",
            "input_audio": {
              "data": "{{item.audio_data}}",
              "format": "wav"
            }
          }
        }
      ]
    }
}

run = await client.create_eval_run_sdk(eval_id, "Audio Emotion Evaluation", data_source)
run_id = run['id']


Created evaluation run for Audio Emotion Evaluation: evalrun_737e7f4df17e432a84fef6cc70a3bff2


## Monitor and Display Results

Poll the evaluation run status until completion, then retrieve and display the results in a DataFrame showing:
- Item IDs
- Grading results (pass/fail status based on emotion matching)
- Expected emotions from the dataset
- Model's identified emotions from the audio inputs

In [6]:
import asyncio
import pandas as pd

while True:
    run = await client.get_eval_run_sdk(eval_id=eval_id, run_id=run_id)
    if run['status'] == "completed":
        output_items_response = await client.get_eval_run_output_items_sdk(
            eval_id=eval_id, run_id=run_id)

        # Get the actual list of items from the response object
        output_items = output_items_response.data if hasattr(output_items_response, 'data') else output_items_response

        # Create DataFrame with safe access to nested fields
        df_data = {
            "id": [],
            "grading_results": [],
            "expected_emotion": [],
            "audio_output": []
        }

        for item in output_items:
            # Convert Pydantic model to dict if needed
            item_dict = item.model_dump() if hasattr(item, 'model_dump') else item
            
            df_data["id"].append(item_dict.get("id", "N/A"))
            results = item_dict.get("results") or []
            result = results[0] if results else {}
            df_data["grading_results"].append({
                "status": result.get("status", "N/A"),
                "score": result.get("score")
            })
            
            # Safely get expected emotion
            datasource_item = item_dict.get('datasource_item', {})
            df_data["expected_emotion"].append(datasource_item.get("expected_emotion", "N/A"))
            
            # Check if audio output exists
            sample = item_dict.get("sample", {})
            output = sample.get("output") or []
            output_transcript = output[0].get("content", "N/A") if output else "N/A"
            df_data["audio_output"].append(output_transcript)

        df = pd.DataFrame(df_data)
        display(df)
        break
    if run['status'] == "failed":
        print("Evaluation run failed:")
        raise RuntimeError(run.get('error', 'Unknown error'))
    print(f"Status: {run['status']}. Waiting...")
    await asyncio.sleep(5)


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


...

Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


Status: in_progress. Waiting...


,id,grading_results,expected_emotion,audio_output
0,1,"{'status': 'completed', 'score': 0.0}",anger,Please wait a moment while I analyze the audio.
1,2,"{'status': 'completed', 'score': 0.0}",disgust,"Sure, let me analyze the emotion from the audi..."
2,3,"{'status': 'completed', 'score': 0.0}",fear,neutral
3,4,"{'status': 'completed', 'score': 0.0}",happy,neutral
4,5,"{'status': 'completed', 'score': 1.0}",neutral,neutral
5,6,"{'status': 'completed', 'score': 0.0}",sad,neutral
6,8,"{'status': 'completed', 'score': 0.0}",anger,neutral
7,9,"{'status': 'completed', 'score': 0.0}",anger,"Sure, please let me analyze the audio to deter..."
